#  RetinaFace

In [ ]:
!pip install -U retinaface_pytorch -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.8 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import zipfile
import random
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from google.colab import drive
from retinaface.pre_trained_models import get_model

# Mount Google Drive
drive.mount('/content/drive')

# ==========================================
# 1. Extract Dataset (If not already extracted)
# ==========================================
# Update this to your exact zip file path in Drive
zip_path = '/content/drive/MyDrive/Celeb-DF-v2.zip'
extract_to = '/content/dataset'

if not os.path.exists(extract_to):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

# Verify directory structure
print("Contents of extract_to:", os.listdir(extract_to))

# ==========================================
# 2. Setup Paths and Model
# ==========================================
# Make sure these folders exist directly under the extract_to path
source_real = os.path.join(extract_to, "Celeb-real")
source_fake = os.path.join(extract_to, "Celeb-synthesis")

base_save_dir = "/content/drive/MyDrive/Deepfake_Project/Balanced_Dataset_V2_RetinaFace"
save_real_dir = os.path.join(base_save_dir, "Real")
save_fake_dir = os.path.join(base_save_dir, "Fake")

os.makedirs(save_real_dir, exist_ok=True)
os.makedirs(save_fake_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize RetinaFace
detector = get_model("resnet50_2020-07-20", max_size=2048, device=device)
detector.eval()

# ==========================================
# NOTE ON QUOTAS
# ==========================================
# Previously this script capped processing at TARGET_QUOTA = 311 videos per
# class (a ~10% few-shot subset). That cap has been removed below so the
# full dataset gets processed. If you ever want a subset again, set
# TARGET_QUOTA back to a number and pass it into process_balanced_set.
FRAMES_PER_VIDEO = 32
IMAGE_SIZE = (380, 380)

# ==========================================
# 3. Helper Functions
# ==========================================
def crop_face_sbi_style(img_rgb, bbox, margin_ratio=0.125):
    """Crop face with a margin equal to 1/8 of the bounding box size."""
    H, W = img_rgb.shape[:2]
    x0, y0, x1, y1 = bbox
    w = x1 - x0
    h = y1 - y0

    w_margin = w * margin_ratio
    h_margin = h * margin_ratio

    y0_new = max(0, int(y0 - h_margin))
    y1_new = min(H, int(y1 + h_margin) + 1)
    x0_new = max(0, int(x0 - w_margin))
    x1_new = min(W, int(x1 + w_margin) + 1)

    return img_rgb[y0_new:y1_new, x0_new:x1_new]

def extract_largest_face(img_rgb):
    """Find faces, select the largest one, crop and resize it."""
    faces = detector.predict_jsons(img_rgb)

    best_bbox = None
    best_area = -1
    for f in faces:
        bbox = f.get('bbox', [])
        if not bbox:
            continue
        x0, y0, x1, y1 = bbox
        area = (x1 - x0) * (y1 - y0)
        if area > best_area:
            best_area = area
            best_bbox = bbox

    if best_bbox is None:
        return None

    cropped = crop_face_sbi_style(img_rgb, best_bbox)
    if cropped.shape[0] == 0 or cropped.shape[1] == 0:
        return None

    resized = cv2.resize(cropped, dsize=IMAGE_SIZE)
    return Image.fromarray(resized)

# ==========================================
# 4. Main Processing Function
# ==========================================
def process_balanced_set(source_folder, save_folder, label, quota=None):
    """
    Process every video in source_folder (or up to `quota` videos if given).
    Designed to be safely re-run after a Colab disconnect: videos that
    already have FRAMES_PER_VIDEO saved frames are skipped, and partially
    processed videos resume from where they left off.
    """
    if not os.path.exists(source_folder):
        print(f"Error: Source folder {source_folder} not found.")
        return 0

    videos = [v for v in os.listdir(source_folder) if v.endswith('.mp4')]
    random.seed(42)
    random.shuffle(videos)

    total_videos = len(videos)
    target = quota if quota is not None else total_videos
    print(f"[{label}] Found {total_videos} videos in source. Target to process: {target}.")

    # Count folders that already have exactly FRAMES_PER_VIDEO frames
    existing_folders = [d for d in os.listdir(save_folder) if os.path.isdir(os.path.join(save_folder, d))]
    success_count = 0
    for d in existing_folders:
        frames = [f for f in os.listdir(os.path.join(save_folder, d)) if f.endswith('.jpg')]
        if len(frames) >= FRAMES_PER_VIDEO:
            success_count += 1

    if success_count > 0:
        print(f"[{label}] Resuming: {success_count} videos already fully processed, skipping those.")

    pbar = tqdm(total=target, desc=f"Processing {label}")
    pbar.update(success_count)

    processed_this_run = 0

    for video_name in videos:
        if success_count >= target:
            break

        video_id = video_name.split('.')[0]
        video_dir = os.path.join(save_folder, video_id)
        os.makedirs(video_dir, exist_ok=True)

        # Check how many frames we already saved for this video
        existing_frames = [f for f in os.listdir(video_dir) if f.startswith('frame_') and f.endswith('.jpg')]
        saved_count = len(existing_frames)

        if saved_count >= FRAMES_PER_VIDEO:
            continue

        video_path = os.path.join(source_folder, video_name)
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames < FRAMES_PER_VIDEO:
            cap.release()
            continue

        # Sample extra frames to handle cases where face detection fails
        max_samples = min(total_frames, FRAMES_PER_VIDEO * 3)
        indices = np.linspace(0, total_frames - 1, max_samples, dtype=int)

        for idx in indices:
            if saved_count >= FRAMES_PER_VIDEO:
                break

            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                continue

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            try:
                face_img = extract_largest_face(frame_rgb)
                if face_img is not None:
                    face_img.save(os.path.join(video_dir, f"frame_{saved_count:02d}.jpg"))
                    saved_count += 1
            except Exception:
                continue

        cap.release()

        processed_this_run += 1

        # Only count as success if we reached the required number of frames
        if saved_count >= FRAMES_PER_VIDEO:
            success_count += 1
            pbar.update(1)

        # Print a live progress line every 10 videos so disconnects/stalls
        # are easy to spot in the Colab log (tqdm alone can be swallowed
        # by Colab's output buffering on long runs).
        if processed_this_run % 10 == 0:
            print(f"[{label}] Progress: {success_count}/{target} videos completed "
                  f"({processed_this_run} attempted this run).")

    pbar.close()
    print(f"[{label}] Done. Total completed: {success_count}/{target}.")
    return success_count

# ==========================================
# 5. Execute Pipeline
# ==========================================
import json

# --- decide which videos are train or test ---
random.seed(42)

all_real_videos = sorted([v.split('.')[0] for v in os.listdir(source_real) if v.endswith('.mp4')])
all_fake_videos = sorted([v.split('.')[0] for v in os.listdir(source_fake) if v.endswith('.mp4')])

train_real_ids = set(random.sample(all_real_videos, min(250, len(all_real_videos))))
train_fake_ids = set(random.sample(all_fake_videos, min(250, len(all_fake_videos))))
train_video_ids = list(train_real_ids | train_fake_ids)

# save to json file (same one for test and train)
train_ids_path = '/content/drive/MyDrive/Deepfake_Project/train_video_ids.json'
with open(train_ids_path, 'w') as f:
    json.dump(train_video_ids, f)
print(f"Train video IDs saved ({len(train_video_ids)} videos): {train_ids_path}")

# --- extract all videos ---
print("--- Starting Full Data Extraction (RetinaFace, SBI-aligned) ---")
real_count = process_balanced_set(source_real, save_real_dir, "Real", quota=590)
fake_count = process_balanced_set(source_fake, save_fake_dir, "Fake", quota=4410)

print(f"\nExtraction Complete!")
print(f"Total Real Videos: {real_count}")
print(f"Total Fake Videos: {fake_count}")

Mounted at /content/drive
Extracting dataset...
Extraction complete.
Contents of extract_to: ['Celeb-real', 'List_of_testing_videos.txt', 'YouTube-real', 'Celeb-synthesis']
Using device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Downloading: "https://github.com/ternaus/retinaface/releases/download/0.01/retinaface_resnet50_2020-07-20-f168fae3c.zip" to /root/.cache/torch/hub/checkpoints/retinaface_resnet50_2020-07-20-f168fae3c.zip


100%|██████████| 96.9M/96.9M [00:02<00:00, 41.1MB/s]
/usr/local/lib/python3.12/dist-packages/torch/hub.py:897: FutureWarning: Falling back to the old format < 1.6. This support will be deprecated in favor of default zipfile format introduced in 1.6. Please redo torch.save() to save it in the new zipfile format.
  return _legacy_zip_load(cached_file, model_dir, map_location, weights_only)


Train video IDs saved (500 videos): /content/drive/MyDrive/Deepfake_Project/train_video_ids.json
--- Starting Full Data Extraction (RetinaFace, SBI-aligned) ---
[Real] Found 590 videos in source. Target to process: 590.
[Real] Resuming: 589 videos already fully processed, skipping those.


...

Processing Fake:  97%|█████████▋| 4287/4410 [7:16:44<22:35, 11.02s/it]

[Fake] Progress: 4287/4410 videos completed (2370 attempted this run).


Processing Fake:  97%|█████████▋| 4297/4410 [7:18:35<20:50, 11.07s/it]

[Fake] Progress: 4297/4410 videos completed (2380 attempted this run).


Processing Fake:  98%|█████████▊| 4305/4410 [7:20:04<19:22, 11.07s/it]